# back-fn-call-with-recipe-args composite — cx24: wrap_forward + call_back_fn round-trip — kwargs survive both halves

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `back-fn-call-with-recipe-args`, `kwargs-pass-through-recipe`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "back-fn-call-with-recipe-args"
DD_ATOM_IDS = ["back-fn-call-with-recipe-args", "kwargs-pass-through-recipe"]
DD_SUBTOPICS = ["Backprop: back fn call with recipe args", "Backprop: Kwargs pass-through"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Composing the back_fn call site with kwargs pass-through

Two atoms join at the back_fn invocation:

- **`kwargs-pass-through-recipe`** — the FORWARD wrapper threads kwargs   (e.g. `dim`, `keepdim`) into BOTH the forward call AND the Recipe.
- **`back-fn-call-with-recipe-args`** — the REVERSE call site splats   those same kwargs back out: `back_fn(grad_out, node.array,   *node.recipe.args, **node.recipe.kwargs)`.

```python
# FORWARD (kwargs-pass-through-recipe):
def wrap_forward_fn(fwd):
    def tensor_func(*args, **kwargs):
        raw = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_raw = fwd(*raw, **kwargs)                       # (1) into call
        out = MiniTensor(out_raw)
        parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
        out.recipe = Recipe(fwd, raw, kwargs, parents)      # (2) into Recipe
        return out
    return tensor_func

# REVERSE (back-fn-call-with-recipe-args):
def call_back_fn(back_fn, grad_out, node):
    return back_fn(
        grad_out,
        node.array,                       # raw torch.Tensor, NOT the MiniTensor
        *node.recipe.args,                # positional from forward
        **node.recipe.kwargs,             # kwargs from forward — same dim/keepdim
    )
```

**The kwargs join forward and reverse.** Without (1), the forward output is wrong (`sum(x, dim=1)` would reduce over the default axis). Without (2), the Recipe loses the `dim` — so at reverse time, `sum_back` has no idea which axis to broadcast back along, and shape errors blow up. Both halves are load-bearing; this composite tests them as ONE round-trip.

### Composite Exercise — wrap_forward + call_back_fn round-trip — kwargs survive both halves

**Atoms exercised together**: `back-fn-call-with-recipe-args`, `kwargs-pass-through-recipe`

Implement TWO halves of the kwargs round-trip:

**1. `cx24_wrap_forward_fn(fwd_fn)`** — closure that returns `tensor_func(*args, **kwargs)` which:
  a. Unboxes each MiniTensor's `.array`; passes non-MiniTensors through.
  b. Calls `fwd_fn(*raw_args, **kwargs)` — kwargs MUST reach the call.
  c. Boxes the result in `MiniTensor(out_raw)` and attaches `out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)` — kwargs MUST be stored.
  d. `parents = {idx: a for idx, a in enumerate(args) if isinstance(a, MiniTensor)}`.

**2. `cx24_call_back_fn(back_fn, grad_out, node)`** — canonical back_fn call:
  ```python
  return back_fn(
      grad_out,
      node.array,             # raw torch.Tensor, NOT the wrapper
      *node.recipe.args,      # positional from forward
      **node.recipe.kwargs,   # kwargs from forward
  )
  ```

**Test round-trip.** Wrap `t.sum`, call it with `dim=1, keepdim=True`. Confirm:
- forward output has shape `(N, 1)` (proves kwargs reached the call).
- `out.recipe.kwargs == {'dim': 1, 'keepdim': True}` (proves they were stored).
- `cx24_call_back_fn` invokes the back_fn with those same kwargs (proves the reverse side reads them).

**Three common bugs the test catches.**
1. Forgetting `**kwargs` in step 2 → forward call uses wrong defaults.
2. Forgetting `kwargs` in step 3 → Recipe has empty kwargs.
3. Passing `node` instead of `node.array` to back_fn → back_fn sees a MiniTensor wrapper, not a raw torch.Tensor.

**A `MiniTensor` and `Recipe` are provided in the test cell.**

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

from dataclasses import dataclass, field
from typing import Any, Callable, Optional

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad=False, recipe=None):
        self.array = array; self.requires_grad = requires_grad
        self.recipe = recipe; self.grad = None

def cx24_wrap_forward_fn(fwd_fn):
    """Return tensor_func that boxes/unboxes and stores kwargs on Recipe."""
    raise NotImplementedError()

def cx24_call_back_fn(back_fn, grad_out, node):
    """Invoke back_fn(grad_out, node.array, *recipe.args, **recipe.kwargs)."""
    raise NotImplementedError()

def _test_cx24():
    # === FORWARD half: kwargs reach BOTH the call and the Recipe ===
    wrapped_sum = cx24_wrap_forward_fn(t.sum)
    x = MiniTensor(t.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]))
    out = wrapped_sum(x, dim=1)
    assert isinstance(out, MiniTensor), 'forward must return MiniTensor'
    assert t.allclose(out.array, t.tensor([6.0, 15.0])), f'sum(dim=1) wrong: {out.array}'
    assert out.recipe is not None, 'Recipe must be attached'
    assert out.recipe.func is t.sum, 'Recipe.func wrong'
    assert out.recipe.kwargs == {'dim': 1}, f'Recipe.kwargs missing dim: {out.recipe.kwargs}'
    assert 0 in out.recipe.parents and out.recipe.parents[0] is x

    # === SECOND kwarg threads through (keepdim) ===
    out2 = wrapped_sum(x, dim=1, keepdim=True)
    assert out2.array.shape == (2, 1), f'keepdim ignored: shape={out2.array.shape}'
    assert out2.recipe.kwargs == {'dim': 1, 'keepdim': True}, out2.recipe.kwargs

    # === empty-kwargs case still works ===
    wrapped_log = cx24_wrap_forward_fn(t.log)
    y = MiniTensor(t.tensor([1.0, t.e, t.e ** 2]))
    out3 = wrapped_log(y)
    assert t.allclose(out3.array, t.tensor([0.0, 1.0, 2.0]), atol=1e-5)
    assert out3.recipe.kwargs == {}, f'empty kwargs case: {out3.recipe.kwargs}'

    # === args on Recipe are RAW torch.Tensors (unboxed) ===
    assert isinstance(out.recipe.args[0], t.Tensor)
    assert not isinstance(out.recipe.args[0], MiniTensor), 'args must be unboxed'

    # === REVERSE half: back_fn receives kwargs from Recipe ===
    received = {}
    def sum_back(grad_out, out_, x_, dim=None, keepdim=False):
        received['grad_out_shape'] = tuple(grad_out.shape)
        received['out_type'] = type(out_)
        received['x_type'] = type(x_)
        received['dim'] = dim
        received['keepdim'] = keepdim
        if dim is None: return grad_out * t.ones_like(x_)
        if not keepdim: grad_out = grad_out.unsqueeze(dim)
        return grad_out.expand_as(x_)

    grad_out = t.tensor([1.0, 1.0])
    result = cx24_call_back_fn(sum_back, grad_out, out)   # out has kwargs={'dim': 1}
    assert received['dim'] == 1, f'kwargs not threaded: dim={received["dim"]}'
    assert received['keepdim'] is False, 'keepdim default reaches back_fn'
    assert received['out_type'] is t.Tensor, (
        f'back_fn must get raw torch.Tensor (node.array), got {received["out_type"]}')
    assert received['x_type'] is t.Tensor, 'recipe.args[0] should be raw'
    assert result.shape == x.array.shape, f'broadcast back via kwargs: {result.shape}'

    # === both kwargs reach back_fn (dim AND keepdim) ===
    received.clear()
    cx24_call_back_fn(sum_back, t.ones(2, 1), out2)        # out2 has dim=1, keepdim=True
    assert received['dim'] == 1, received
    assert received['keepdim'] is True, received

    # === arbitrary kwarg (scale) round-trips correctly ===
    def fake_op(x, *, scale=1.0): return x * scale
    wrapped_fake = cx24_wrap_forward_fn(fake_op)
    z = MiniTensor(t.tensor([2.0, 3.0]))
    out_fake = wrapped_fake(z, scale=4.0)
    assert t.allclose(out_fake.array, t.tensor([8.0, 12.0])), out_fake.array
    assert out_fake.recipe.kwargs == {'scale': 4.0}
    spy = {}
    def fake_back(grad_out, out_, x_, *, scale=1.0):
        spy['scale'] = scale
        return grad_out * scale
    g = cx24_call_back_fn(fake_back, t.ones(2), out_fake)
    assert spy['scale'] == 4.0, f'scale must reach back_fn: {spy}'
    assert t.allclose(g, t.tensor([4.0, 4.0])), g
    _dd_passed.add('cx24')

_test_cx24()

<details><summary>Show solution — cx24</summary>

```python
def cx24_wrap_forward_fn(fwd_fn):
    def tensor_func(*args, **kwargs):
        # Unbox MiniTensors → raw torch.Tensors; pass-through non-MiniTensors.
        raw_args = tuple(
            a.array if isinstance(a, MiniTensor) else a for a in args
        )
        # (1) forward MUST receive kwargs
        out_raw = fwd_fn(*raw_args, **kwargs)
        # parents-dict-by-argidx (composed elsewhere) — keep original idx.
        parents = {
            idx: a for idx, a in enumerate(args) if isinstance(a, MiniTensor)
        }
        out = MiniTensor(out_raw)
        # (2) kwargs MUST also be stored on the Recipe — for the reverse call.
        out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func

def cx24_call_back_fn(back_fn, grad_out, node):
    # Canonical four-channel invocation — note BOTH splats.
    return back_fn(
        grad_out,
        node.array,                     # raw torch.Tensor, NOT MiniTensor
        *node.recipe.args,              # positional from forward
        **node.recipe.kwargs,           # kwargs from forward (dim, keepdim, ...)
    )
```

**Two atoms, one round-trip.** `kwargs-pass-through-recipe` is the FORWARD half — kwargs reach (a) the actual call and (b) the stored Recipe. `back-fn-call-with-recipe-args` is the REVERSE half — kwargs are splatted out of the Recipe and into the back_fn. Drop either half and the round-trip breaks; only the COMPOSITION yields a working `sum`/`mean`/`max` autograd op.

**Three splats matter.** `*raw_args` unboxes the positional inputs; `*node.recipe.args` re-splats them on the way back; `**node.recipe.kwargs` re-splats the keyword args. Drop any of the three and a real autograd implementation fails on shape mismatches deep inside the back_fn.

**`node.array`, not `node`.** Back_fns operate on raw torch tensors so they can do tensor math without unboxing in each body. Passing the MiniTensor wrapper would force every back_fn body to start with `out.array` — defeating the whole point of the unboxing wrapper.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx24'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx24',
        'subtopics': ["Backprop: back fn call with recipe args", "Backprop: Kwargs pass-through"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()